<a href="https://colab.research.google.com/github/lili-codelab/comp-linguistics/blob/main/%D0%A1%D0%B0%D0%B2%D1%87%D0%B5%D0%BD%D0%BA%D0%BE_%D0%A8%D0%B8%D0%BB%D0%BE%D0%B2%D1%81%D0%BA%D0%B8%D0%B9_%D0%A2%D1%83%D0%B9%D0%B3%D1%83%D0%BD%D0%BE%D0%B2%D0%B0_%D0%9F%D0%BE%D1%81%D1%82%D1%80%D0%BE%D0%B5%D0%BD%D0%B8%D0%B5_RAG_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D1%8B_%D1%81_%D0%B8%D1%81%D0%BF%D0%BE%D0%BB%D1%8C%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5%D0%BC_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Тема:** RAG (Retrieval-Augmented Generation) с фреймворком LangChain


Разработка RAG-пайплайна


**Задачи:**
* Загрузить набор текстовых документов (например, статей из датасета arXiv Dataset: https://www.kaggle.com/datasets/Cornell-University/arxiv)
* Разбить текст на чанки с помощью Langchain text splitter
* Создать векторный индекс с помощью FAISS и sentence-transformers
* Реализовать langchain-цепочку, которая производим семантический поиск и формирует промпт для LLM (локальной или через Groq/OpenRouter)
* Протестировать систему на нескольких вопросах, оценить качество ответов


**Библиотеки:** langchain, huggingface, faiss-cpu, sentence-transformers

**Ожидаемый результат:** Colab-ноутбук с рабочим прототипом наукоёмкой (например, разработанной на основе текстов ArXiv) RAG-системы, примерами её ответов и качественным анализом, представленным в текстовых блоках


## Загрузить набор текстовых документов

### Датасет с метаданными к статьям

In [119]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "arxiv-metadata-oai-snapshot.json"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "Cornell-University/arxiv",
  file_path,
  pandas_kwargs={"lines": True, "nrows": 200000},
)

# в датасете встречаются id разного формата, приводим их к одному в виде XXXX.XXXX
def normalize_arxiv_id(x):
    x = str(x).strip()

    if "." in x:
        left, right = x.split(".", 1)
        left = left.zfill(4)
        right = right.zfill(4)
        return f"{left}.{right}"

    return x

df["id"] = df["id"].apply(normalize_arxiv_id)

/tmp/ipykernel_38112/1629828404.py:6: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'arxiv' dataset.


In [120]:
df.head(2)

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
0,0704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",Calculation of prompt diphoton production cros...,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturba...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky,..."
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"


### Подбор статей

 https://arxiv.org/abs/{id}: посмотреть страницу статьи с абстрактом

 https://arxiv.org/pdf/{id}: скачать

In [121]:
import re

topic_keywords = [
    "retrieval",
    "question answering",
    "language model",
    "transformer",
    "prompt",
    "rag"
]

# фильтр по категориям arXiv
mask_cat = df["categories"].fillna("").str.contains(
    r"\bcs\.(CL|AI|LG|IR)\b", regex=True
)

# фильтр по ключевым словам (заголовок + аннотация)
text_for_filter = (
    df["title"].fillna("") + " " + df["abstract"].fillna("")
).str.lower()

mask_kw = text_for_filter.str.contains("|".join(map(re.escape, topic_keywords)))

# итоговая выборка
selected_df = (
    df[mask_cat & mask_kw]
    .drop_duplicates(subset="id")
    .head(120)
    .copy()
)

ids = selected_df["id"].tolist()

print("Количество отобранных статей:", len(ids))
selected_df[["id", "title", "categories"]].head(120)

/tmp/ipykernel_38112/2979700595.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_cat = df["categories"].fillna("").str.contains(


Количество отобранных статей: 120


,id,title,categories
953,0704.0954,Sensor Networks with Random Links: Topology De...,cs.IT cs.LG math.IT
1019,0704.0102,The on-line shortest path problem under partia...,cs.LG cs.SC
1027,0704.1028,A neural network approach to ordinal regression,cs.LG cs.AI cs.NE
2901,0704.2902,Recommending Related Papers Based on Digital L...,cs.DL cs.IR
3394,0704.3395,General-Purpose Computing on a Semantic Networ...,cs.AI cs.PL
...,...,...,...
119261,0904.2595,A Methodology for Learning Players' Styles fro...,cs.AI cs.LG
120135,0904.3469,Toggling operators in computability logic,cs.LO cs.AI math.LO
120367,0904.3701,Semantic Social Network Analysis,cs.AI
120707,0904.4041,Content-Based Sub-Image Retrieval with Relevan...,cs.DB cs.IR


### Загрузка

In [122]:
!pip install langchain_community langchain_text_splitters pypdf -q

In [123]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [124]:
from pathlib import Path
import requests
import time

pdf_dir = Path("arxiv_pdfs")
pdf_dir.mkdir(exist_ok=True)

meta_map = selected_df.set_index("id")[["title", "categories", "abstract"]].to_dict(orient="index")

docs = []
MAX_PDFS = 120

for paper_id in ids[:MAX_PDFS]:
    url = f"https://arxiv.org/pdf/{paper_id}.pdf"
    pdf_path = pdf_dir / f"{paper_id}.pdf"

    try:
        # если файла нет локально — скачиваем
        if not pdf_path.exists():
            response = requests.get(
                url,
                timeout = 40,
                headers = {"User-Agent": "Mozilla/5.0"}
            )
            response.raise_for_status()
            pdf_path.write_bytes(response.content)

        # читаем PDF, разбиваем по страницам + фильтр
        loader = PyPDFLoader(str(pdf_path))
        paper_docs = loader.load()
        paper_docs = [
            doc for doc in paper_docs
            if len(doc.page_content.strip()) > 50
            and "/uni" not in doc.page_content
        ]

        # добавляем метаданные к каждой странице
        meta = meta_map.get(paper_id, {})
        for doc in paper_docs:
            doc.metadata["paper_id"] = paper_id
            doc.metadata["title"] = meta.get("title", "")
            doc.metadata["categories"] = meta.get("categories", "")
            doc.metadata["abstract"] = meta.get("abstract", "")

        docs.extend(paper_docs)
        print(f"OK: {paper_id} | pages: {len(paper_docs)}")

        time.sleep(1)  # ограничиваем скорость между запросами arXiv

    except Exception as e:
        print(f"Ошибка для {paper_id}: {e}")

print("Всего Document-объектов:", len(docs))

# очистка перед чанкованием
for doc in docs:
    text = doc.page_content
    text = re.sub(r"-\s+", "", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\b\d+\s+\d+\s+\d+.*?\b", "", text)
    text = text.replace("\n", " ")
    doc.page_content = text.strip()

OK: 0704.0954 | pages: 30
OK: 0704.0102 | pages: 18
OK: 0704.1028 | pages: 8
OK: 0704.2902 | pages: 2
OK: 0704.3395 | pages: 34
OK: 0704.3433 | pages: 20
OK: 0705.0593 | pages: 10
OK: 0705.0693 | pages: 6
OK: 0705.0751 | pages: 5
OK: 0705.0111 | pages: 8
OK: 0705.1673 | pages: 6
OK: 0705.1999 | pages: 14
OK: 0705.2106 | pages: 5
OK: 0705.0231 | pages: 20
OK: 0705.2516 | pages: 7
OK: 0705.4566 | pages: 16
OK: 0705.4606 | pages: 11
OK: 0706.0022 | pages: 8
OK: 0706.1179 | pages: 8
OK: 0706.4375 | pages: 15
OK: 0707.3087 | pages: 15
OK: 0707.3457 | pages: 8
OK: 0707.3559 | pages: 166
OK: 0707.3575 | pages: 16
OK: 0708.0694 | pages: 14
OK: 0708.2788 | pages: 10
OK: 0708.3259 | pages: 16
OK: 0708.4149 | pages: 13
OK: 0709.2562 | pages: 13
OK: 0709.4669 | pages: 30
OK: 0710.0228 | pages: 6
OK: 0710.1481 | pages: 6
OK: 0710.2889 | pages: 22
OK: 0710.4516 | pages: 3
OK: 0711.1038 | pages: 8
OK: 0711.2023 | pages: 30
OK: 0711.2801 | pages: 31
OK: 0711.3128 | pages: 6
OK: 0711.3449 | pages: 5
OK

OK: 0803.0053 | pages: 8
OK: 0803.1716 | pages: 35
OK: 0803.0222 | pages: 11
OK: 0803.2306 | pages: 48
OK: 0803.3693 | pages: 19
OK: 0803.3838 | pages: 16
OK: 0804.0143 | pages: 15
OK: 0804.2057 | pages: 21
OK: 0804.3575 | pages: 27
OK: 0804.3599 | pages: 8
OK: 0805.2303 | pages: 17
OK: 0805.2362 | pages: 7
OK: 0805.3521 | pages: 30


OK: 0805.3802 | pages: 4
OK: 0805.4369 | pages: 32
OK: 0806.0784 | pages: 5
OK: 0806.1156 | pages: 5
OK: 0806.1636 | pages: 34
OK: 0806.3765 | pages: 19
OK: 0806.4802 | pages: 15
OK: 0806.4921 | pages: 26
OK: 0807.0023 | pages: 12
OK: 0807.1005 | pages: 44
OK: 0807.4198 | pages: 83
OK: 0808.0521 | pages: 43
OK: 0808.0973 | pages: 22
OK: 0808.1125 | pages: 9
OK: 0809.0036 | pages: 17
OK: 0809.0406 | pages: 10
OK: 0809.0041 | pages: 5
OK: 0809.0753 | pages: 5
OK: 0809.0755 | pages: 4
OK: 0809.1241 | pages: 254
OK: 0809.0317 | pages: 17
OK: 0809.0453 | pages: 17
OK: 0809.4668 | pages: 16
OK: 0809.4834 | pages: 15
OK: 0810.2764 | pages: 6
OK: 0810.4616 | pages: 11
OK: 0810.5407 | pages: 289
OK: 0811.0126 | pages: 9
OK: 0811.4458 | pages: 14
OK: 0811.4717 | pages: 11
OK: 0812.0698 | pages: 13
Ошибка для 0812.0885: 404 Client Error: Not Found for url: https://arxiv.org/pdf/0812.0885
OK: 0812.2785 | pages: 4
OK: 0901.0358 | pages: 7
OK: 0901.0213 | pages: 18
OK: 0901.0359 | pages: 31
OK: 0901

## Разбить на чанки

In [188]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 300,
    length_function = len,
    separators = ["\n\n", "\n", ". ", "! ", "? ", " ", ""],
    add_start_index = True
)

chunks = text_splitter.split_documents(docs)

chunks = [
    chunk for chunk in chunks
    if len(chunk.page_content.strip()) > 100
    and "/uni" not in chunk.page_content
]

# для быстрого теста
# chunks = chunks[:1000]

print(f"Всего документов: {len(docs)}")
print(f"Всего чанков после разбиения: {len(chunks)}")
print(f"Среднее количество чанков на документ: {len(chunks)/len(docs):.1f}")

Всего документов: 2629
Всего чанков после разбиения: 6218
Среднее количество чанков на документ: 2.4


## Создать векторный индекс с помощью faiss-cpu и sentence-transformers

In [189]:
!pip install faiss-cpu sentence-transformers langchain-huggingface -q

In [190]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

vectorstore = FAISS.from_documents(chunks, embedding_model)

print(f"Обработано чанков: {vectorstore.index.ntotal}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Обработано чанков: 6218


## Реализовать цепочку

In [191]:
!pip install langchain_core langchain_classic -q

In [192]:
!pip install langchain-openai -q

In [193]:
import json
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate

In [194]:
# с HuggingFace возникали трудности, поэтому было решено работать с OpenRouter

GEN_MODEL_ID = "openai/gpt-oss-120b:free"
TOP_K = 4
QUESTION = "Что представляет собой метод FastMap?"
PROMPT = PromptTemplate(
    input_variables=["context", "input"],
    template="""Ты — ассистент, отвечающий на вопросы на основе научных статей. Используй предоставленные данные, чтобы ответить на вопрос корректно.
Контекст:
{context}
Вопрос: {input}
Ответ:"""
)
TASK = "conversational"

In [195]:
from google.colab import userdata
from langchain_openai import ChatOpenAI

OR_KEY = userdata.get('PROJECT')

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
llm = ChatOpenAI(
    api_key = OR_KEY,
    base_url = "https://openrouter.ai/api/v1",
    model = "openai/gpt-oss-120b:free",
)

In [196]:
def clip_text(text, threshold=100):
    return f"{text[:threshold]}..." if len(text) > threshold else text

In [197]:
question_answer_chain = create_stuff_documents_chain(llm, PROMPT)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
resp_dict = rag_chain.invoke({"input": QUESTION})

if not resp_dict.get("answer"):
    print("Недостаточно информации, чтобы ответить на вопрос.")

clipped_answer = clip_text(resp_dict["answer"], threshold=1000)
print(f"Question:\n{resp_dict['input']}\n\nAnswer:\n{clipped_answer}")
for i, doc in enumerate(resp_dict["context"]):
    print()
    print(f"Source {i + 1}:")
    print(f"  text: {json.dumps(clip_text(doc.page_content, threshold=500))}")
    meta_keys = ["title", "source", "page", "page_label", "categories", "paper_id"] # для компактности вывода, иначе выходит очень большой объем информации
    meta_str = ", ".join(f"{k} - {doc.metadata.get(k,'')}" for k in meta_keys)
    print(f"  metadata: {meta_str}")

Question:
Что представляет собой метод FastMap?

Answer:
**FastMap** — это быстрый алгоритм **встраивания (embedding) объектов из произвольного метрического пространства в евклидову пространство низкой размерности** (обычно k = 2–5), при котором сохраняются их взаимные расстояния с приемлемой точностью.  

### Ключевые идеи метода  

| Шаг | Описание |
|-----|----------|
| **1. Выбор опорных (pivot) объектов** | На каждом рекурсивном уровне выбираются два «дальних» объекта \(p\) и \(q\) (пивоты). Их расстояние \(d(p,q)\) используется как масштабный фактор. |
| **2. Вычисление координаты вдоль текущей оси** | Для любого объекта \(o\) его координата \(x_o\) по первой (или текущей) оси определяется из известного расстояния до пивотов по формуле Пифагора: <br> \[
x_o = \frac{d(o,p)^2 + d(p,q)^2 - d(o,q)^2}{2\,d(p,q)} .
\] |
| **3. Обновление расстояний (рекурсия)** | После получения координаты \(x_o\) вычитается её вклад из исходных расстояний, получая «остаточные» расстояния, которые зате

## Протестировать на нескольких примерах, оченить качество

In [201]:
QUESTIONS = [
    "Какие примеры применения упоминаются вместе с FastMap?",
    "Как в статьях описывается обработка естественного языка или текстовых данных?",
    "Как испечь пирог?",
    "Какие алгоритмы машинного обучения упоминаются в статьях?"
]

print("Тестирование RAG-системы с использованием LangChain")

for q in QUESTIONS:
    print("\nВопрос:", q)

    try:
        resp = rag_chain.invoke({"input": q})
    except Exception as e:
        print("Ошибка при вызове модели:", e)
        resp = {"answer": "Недостаточно информации, чтобы ответить на вопрос.", "context": []}

    answer = resp.get("answer")
    if not answer or answer.strip() == "":
        answer = "Недостаточно информации, чтобы ответить на вопрос."

    print("Ответ:", answer[:1500], "...\n")

    for i, doc in enumerate(resp.get("context", [])):
        meta_str = ", ".join(f"{k} - {v}" for k,v in doc.metadata.items() if k in ["title","source","page","page_label"])
        print(f"Source {i+1}: {meta_str}")
        print("Text:", doc.page_content[:300], "...\n")


Тестирование RAG-системы с использованием LangChain

Вопрос: Какие примеры применения упоминаются вместе с FastMap?
Ответ: В предоставленном отрывке текста **FastMap** не упоминается, поэтому никаких примеров его применения в данном контексте нет. Если нужны сведения о том, где обычно используют FastMap (например, для быстрой проекции многомерных данных в низко‑размерное пространство, ускорения поиска похожих объектов, визуализации больших наборов точек и т.п.), их следует искать в других частях статьи или в отдельной литературе, где описывается этот метод. ...

Source 1: title - Effectively Searching Maps in Web Documents, source - arxiv_pdfs/0901.3939.pdf, page - 1, page_label - 2
Text: . This is the ﬁrst such system to fully parse a variety of actual diagrams drawn from the research literature. Digmap system4 [6] is a geographic IR system based on the historical digitized maps. Our work is different from those we described above. In their approaches, the information is dynamically .

В ходе тестирования модели были получены ответы разного качества.

На первый вопрос, "Какие примеры применения упоминаются вместе с FastMap?", модель ответила:

> Ответ: В приведённом фрагменте текста **FastMap** не упоминается, поэтому никаких примеров его применения в данном контексте не дано. Единственное, что встречается, — это ссылка на систему **FASTUS** (Appelt et al., 1993) и описание её использования для построения шаблонов‑правил, но это не относится к FastMap. Таким образом, на основании предоставленных данных нельзя назвать примеры применения FastMap.

> То есть, что FastMap не упоминается в предоставленных документах, и, следовательно, примеры его применения отсутствуют. Тем не менее, в блоке реализации цепочки модель дала довольно развернутый ответ о том, что такое **FastMap**, как работает и некоторый обрывок (незаконченной дополнительной информации из-за ограничения количества символов для вывода). Возможно, это произошло после некоторых изменений (тестирование другой модели и последующий выбор действующей), или в представленной информации не было информации конкретно по применению.


На второй вопрос, "Как в статьях описывается обработка естественного языка или текстовых данных?", был получен довольно подробный и структурированный ответ:

> Ответ: В представленных фрагментах обработка естественного языка (ОЕЯ) и работа с текстовыми данными описываются тремя типологически разными подходами, каждый из которых подчёркивает свои цели и используемые инструменты.

| Статья / Фрагмент | Основной метод обработки | Ключевые шаги и инструменты | Что считается «обработкой» |
|-------------------|--------------------------|----------------------------|---------------------------|
| **McAllester & Givan (логический фрагмент)** | Формальная логика – синтаксический анализ формул. | • Определяются *formula sets* (множества формул), которые «determine existentials». <br>• Предлагается *syllogistic‑like system* (силлогистическая система) для вывода. <br>• Доказана **полнота** системы относительно выбранных множеств формул. | Обработка заключается в преобразовании естественно‑языковых утверждений в формальные логические формы и последующем выводе новых фактов (существования) с помощью доказательной системы. |
| **Исследование тональных индексов в русском языке** | Корпусный, аннотационный и просодический анализ. | • Сбор **корпуса спонтанной речи** на русском. <br>• **Аннотирование** (разметка) просодических единиц (тональные индексы, границы слов, мелкие просодические единицы). <br>• Разработка **полуавтоматических heuristics** для предсказания границ слов и оценки вклада тональной информации. <br>• Использование **иерархической представления просодической конституентности** (уровни: мелкие единицы → слова → фразы). | Обработка включает ...

Информация была взята из одной статьи, модель обобщила информацию из чанков, выделила ключевые моменты и структурировала. Вероятно, при увеличении размера выводимого ответа получилось бы гораздо больше информации в таблице.


Третий вопрос был задан с целью оценить ответ на случай, когда информация отсутствует: "Как испечь пирог?" На что был предоставлен удовлетворительный ответ:

> Ответ: В предоставленных вам данных содержится информация о лексических ресурсах, классификациях и вопросах транслитерации, но никаких сведений о рецептах, технологиях приготовления или кулинарных рекомендациях, в частности о том, как испечь пирог, нет. Поэтому на основе этих материалов я не могу дать ответ на ваш вопрос. Если понадобится информация о выпечке, её следует искать в кулинарных источниках или специализированных научных публикациях, посвящённых пищевой технологии.

Иными словами - четкое указание на то, что в материалах нет нужных сведений и даже предложение о других источниках. Система распознала вопрос вне контекста  и в этом случае правильно применила инструкцию "Недостаточно информации, чтобы ответить на вопрос."


Последний вопрос, "Какие алгоритмы машинного обучения упоминаются в статьях?", также был распознан верно:

> Ответ: В приведённых отрывках из статей явно упоминаются два подхода, относящиеся к машинному обучению:

| Алгоритм / метод | Где упомянут | Краткое описание |
|------------------|--------------|------------------|
| **N‑gram‑based text categorization** | Cavnar & Trenkle 1994 (см. ссылка в тексте) | Классификация текстов на основе частотных n‑грамм; один из классических методов машинного обучения для построения моделей «меш‑оф‑слов». |
| **Cluster‑specific named‑entity transliteration** | F. Huang (см. ссылка в тексте) | Метод трансляции имён собственных, построенный с учётом кластеров (групп) сущностей; использует обучение на примерах внутри каждого кластера. |

> Других конкретных названий алгоритмов машинного обучения в данном фрагменте не обнаружено. (Упоминания вроде «MEP», «RS», «DM» относятся к обозначениям экспериментальных конфигураций, а не к названиям известных алгоритмов.)

В ответе представлены не все методы, возможно, не хватает статей по теме NLP. Тем не менее, пару методов перечислено, дается краткое описание, укащано упоминание в конкретных статьях.


**Общая оценка**

На вопросы строго по тематике статей модель дает содержательные ответы, в основном опираясь на контекст. Ответы хорошо структурированы (помимо общей информации, предоставляются таблицы). При действительном отсутствии информации ответ соответствует инструкции.
В случае с ответом на первый вопрос - не совсем очевидна причина, можно сказать, ошибки. Также корректные вопросы в основном опираются на минимальное количесвто статей. Другие статьи (тем не менее схожие по тематике и полезные в некотором плане) практически остаются без внимания.

Вывод: RAG-система работоспособна (находит релевантные фрагменты, извлекает из них информацию и формирует ответы), однако полнота охвата материалов немного ограничена. Для улучшения, возможно, стоит расширить набор статей, либо поэксперементировать с настройками.




---



# Критерии оценки

Работа проверяется по следующим критериям (максимум 10 баллов):

### Загрузка и подготовка данных (2 балла)
- [ ] 0.5 балла: выбран критерий подбора материалов
- [ ] 0.5 балла: загружено не менее 100 записей/статей
- [ ] 0.5 балла: тексты успешно извлечены из источника
- [ ] 0.5 балла: данные приведены к формату, пригодному для чанкинга (очистка, объединение полей)

### Чанкинг (2 балла)
- [ ] 0.5 балла: выбран подходящий тип сплиттера (RecursiveCharacterTextSplitter, HTMLHeaderTextSplitter и т.д.)
- [ ] 0.5 балла: обоснован выбор размера чанка и перекрытия (например, "512 токенов, overlap 20% для сохранения контекста")
- [ ] 0.5 балла: чанки созданы и не содержат явных артефактов (оборванных слов)
- [ ] 0.5 балла: количество чанков соответствует ожидаемому (не 1 и не 100500 на документ)

### Векторное хранилище (1 балл)
- [ ] 0.5 балла: выбрана адекватная эмбеддинг-модель (например, all-MiniLM-L6-v2 для русского/английского)
- [ ] 0.5 балла: индекс создан


### Реализация цепочки (3 балла)
- [ ] 0.5 балла: выбрана LLM
- [ ] 1 балл: промпт, QUESTION, TASK составлены корректно
- [ ] 0.5 балла: обоснован заданный TOP_K
- [ ] 0.5 балла: ответ генерируется на основе найденных чанков (видно по содержанию)
- [ ] 0.5 балла: обработан случай отсутствия информации в контексте

### Тестирование и анализ (2 балла)
- [ ] 0.5 балла: задано минимум 3 разнотипных вопроса (фактический, обобщающий, уточняющий)
- [ ] 0.5 балла: для каждого вопроса показан и проанализирован ответ
- [ ] 0.5 балла: в анализе указано, какие чанки использовались и почему
- [ ] 0.5 балла: сделан вывод о качестве работы системы (что получилось, что нет, гипотезы почему)
